<div align="center">

# Text-to-SQL with BART and GPT-2

### Comparing Encoder–Decoder and Decoder-Only Transformer Architectures

</div>

---

## Project Overview

Text-to-SQL is a natural language generation task in which a model translates a user question into a valid SQL query based on a given database schema.

This project investigates two different Transformer architectures for this task:

- **BART** as an Encoder–Decoder model
- **GPT-2** as a Decoder-Only model

Both models are fine-tuned on a synthetic Text-to-SQL dataset and evaluated under the same experimental setting.

The project focuses on the complete Text-to-SQL pipeline, including:

- Dataset preparation and preprocessing
- Schema-aware input construction
- BART fine-tuning
- GPT-2 fine-tuning
- SQL generation
- Exact Match evaluation
- SQL normalization
- Error analysis
- Architectural comparison

A particular focus is placed on how the two architectures represent the natural-language question, database schema, and target SQL query.

---

## Project Pipeline

The project is organized into four main stages:

1. **Dataset Preparation and Preprocessing**
2. **Encoder–Decoder Fine-Tuning with BART**
3. **Decoder-Only Fine-Tuning with GPT-2**
4. **Evaluation and Comparative Analysis**

---

## Text-to-SQL Considerations

Text-to-SQL generation is highly sensitive to output formatting and structural differences.

Even small variations in:

- SQL keyword casing
- Whitespace
- Semicolons
- Clause ordering
- Schema references

can affect exact-match evaluation.

For this reason, the project evaluates generated queries using both raw and normalized representations to provide a more consistent comparison between predictions and reference SQL queries.

The two model architectures also require different input formulations.

For **BART**, the natural-language question and database schema are encoded as input, while the SQL query is generated independently by the decoder.

For **GPT-2**, the schema, question, and SQL target are represented within a single autoregressive sequence. During training, non-target tokens are masked so that the loss is calculated only over the SQL generation portion.

---

# 1. Dataset Preparation and Preprocessing

## Dataset

The project uses the **Gretel Synthetic Text-to-SQL** dataset available through Hugging Face.

The dataset contains natural-language questions paired with database schemas and corresponding SQL queries.

The main fields used in the project are:

- `question` — the natural-language request
- `schema` — the database structure available to the model
- `query` — the target SQL statement

The dataset is inspected before training to understand the relationship between user questions, database schemas, and target SQL queries.

## Data Preparation

The preprocessing pipeline includes:

- Loading the training and test splits
- Inspecting dataset structure and sample records
- Standardizing column names
- Creating training and development subsets
- Limiting dataset size to make experimentation computationally manageable
- Preparing the resulting data for model-specific tokenization

A sample instance is also examined to illustrate how the natural-language question, schema information, and target SQL query are connected.

## Dataset Challenges

Text-to-SQL presents several challenges for sequence-generation models.

The model must simultaneously learn to:

- Understand the intent of the natural-language question
- Identify relevant tables and columns from the schema
- Generate syntactically valid SQL
- Preserve relationships between schema elements
- Produce the correct SQL structure and conditions

As schema complexity increases, the model must reason over a larger set of possible tables, columns, and relationships while maintaining valid SQL syntax.

In [1]:
import pandas as pd
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
import time
import random
import re
import numpy as np
import torch
import sqlparse
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)


In [ ]:
dataset = load_dataset("philschmid/gretel-synthetic-text-to-sql")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/737 [00:00<?, ?B/s]

synthetic_text_to_sql_train.snappy.parqu(…):   0%|          | 0.00/32.4M [00:00<?, ?B/s]

synthetic_text_to_sql_test.snappy.parque(…):   0%|          | 0.00/1.90M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5851 [00:00<?, ? examples/s]

In [3]:
train_ds = dataset["train"]
test_ds  = dataset["test"]

In [ ]:
train_df = train_ds.to_pandas()
test_df  = test_ds.to_pandas()

In [5]:
train_df[:1]

,id,domain,domain_description,sql_complexity,sql_complexity_description,sql_task_type,sql_task_type_description,sql_prompt,sql_context,sql,sql_explanation
0,5097,forestry,Comprehensive data on sustainable forest manag...,single join,"only one join (specify inner, outer, cross)",analytics and reporting,"generating reports, dashboards, and analytical...",What is the total volume of timber sold by eac...,"CREATE TABLE salesperson (salesperson_id INT, ...","SELECT salesperson_id, name, SUM(volume) as to...","Joins timber_sales and salesperson tables, gro..."


In [ ]:
train_df = train_df.rename(columns={"sql_prompt": "question", "sql_context": "schema", "sql": "query"})
test_df  = test_df.rename(columns={"sql_prompt": "question", "sql_context": "schema", "sql": "query"})

# negah dashtane soton haye morede niaz
train_df = train_df[["question", "schema", "query"]]
test_df  = test_df[["question", "schema", "query"]]


In [ ]:
# Create the train/dev split

train_df, dev_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42
)

In [ ]:
# Use fixed-size subsets for faster experimentation

TRAIN_SIZE = 20000
DEV_SIZE   = 2000
TEST_SIZE  = 2000

train_ds_small = train_df.sample(n=TRAIN_SIZE, random_state=42)
dev_ds_small = dev_df.sample(n=DEV_SIZE, random_state=42)
test_ds_small = test_df.sample(n=TEST_SIZE, random_state=42)

In [9]:
train_ds_small[:1]

,question,schema,query
56958,"List the machine IDs, types, and maintenance s...","CREATE TABLE machines (machine_id INT, type TE...","SELECT machines.machine_id, machines.type, mac..."


# 2. Encoder–Decoder Fine-Tuning with BART

## BART for Text-to-SQL

BART is an Encoder–Decoder Transformer architecture designed for sequence-to-sequence tasks.

In this project, BART is used to map a structured natural-language input consisting of the user question and database schema to a target SQL query.

The encoder processes the complete input representation, while the decoder autoregressively generates the corresponding SQL sequence.

This architecture is particularly suitable for Text-to-SQL because the task naturally involves transforming one sequence:

```text
Question + Database Schema
```

into another:

```text
SQL Query
```

---

## Input Construction

Each training example combines the natural-language question and database schema into a single input sequence.

The input is structured so that the model can distinguish between the user request and the available database structure before generating the target SQL query.

The preprocessing pipeline includes:

* Combining the question and schema into a unified model input
* Tokenizing the input sequence
* Creating attention masks
* Tokenizing the target SQL query
* Preparing target labels for sequence-to-sequence training

---

## BART Dataset Pipeline

A dedicated dataset pipeline is used to prepare the Text-to-SQL samples for BART.

For each sample:

1. The question and schema are converted into the model input format.
2. The input sequence is tokenized.
3. The corresponding SQL query is tokenized as the target sequence.
4. Attention masks and labels are prepared for training.

The resulting dataset is then used with the Hugging Face `Seq2SeqTrainer`.

---

## Model Training

The BART model is fine-tuned on a subset of **20,000 training samples**.

The training process tracks:

* Training loss
* Training runtime
* Learning rate
* Batch size
* Number of epochs
* Maximum input and target sequence lengths

These settings are kept consistent throughout the experiment to support reproducibility and later comparison with the GPT-2 model.

---

## Evaluation Metrics

The generated SQL queries are evaluated using two Exact Match metrics.

### Raw Exact Match

Raw Exact Match directly compares the generated SQL query with the reference query as strings.

A prediction is counted as correct only when both strings match exactly.

This metric is strict and may treat semantically identical SQL queries as different because of superficial formatting differences.

Examples include:

* Uppercase vs. lowercase SQL keywords
* Extra whitespace
* Trailing semicolons
* Minor formatting differences

---

### Normalized Exact Match

Normalized Exact Match applies lightweight normalization before comparing the generated and reference queries.

The normalization process reduces differences caused by formatting rather than SQL content.

The normalization includes operations such as:

* Converting text to lowercase
* Removing unnecessary whitespace
* Standardizing SQL formatting
* Removing trailing semicolons

This provides a more robust comparison when predictions differ only in surface-level formatting.

---

## Evaluation on Development and Test Sets

The fine-tuned BART model is evaluated independently on both the development and test subsets.

For each split, the following metrics are reported:

* Raw Exact Match
* Normalized Exact Match

This makes it possible to measure both strict string-level correctness and correctness after SQL normalization.

---

## Qualitative Error Analysis

In addition to aggregate evaluation metrics, several randomly selected development examples are inspected manually.

For each example, the analysis includes:

* Natural-language question
* Database schema
* Generated SQL query
* Reference SQL query
* Observed error type

The qualitative analysis helps identify common failure patterns such as:

* Incorrect table or column selection
* Missing or incorrect conditions
* Invalid SQL structure
* Incorrect aggregation
* Incomplete query generation
* Formatting-only differences


In [ ]:
# Convert sampled DataFrames to Hugging Face datasets

train_ds_small = Dataset.from_pandas(train_ds_small)
dev_ds_small   = Dataset.from_pandas(dev_ds_small)
test_ds_small  = Dataset.from_pandas(test_ds_small)

In [ ]:
# func baraye sakhte inpute BART

def build_bart_input(question, schema):
    """
    Construct the input prompt for BART by combining the database schema
    and natural-language question.

    Args:
        question (str): Natural-language question to be translated into SQL.
        schema (str): Database schema associated with the question.

    Returns:
        str: Formatted Text-to-SQL prompt for the model.
    """
    
    return f"Translate to SQL.\nSchema:\n{schema}\nQuestion:\n{question}\nSQL:"



In [12]:
# meghdare dahi avalie

MODEL_NAME = "facebook/bart-base"
MAX_SOURCE_LEN = 512
MAX_TARGET_LEN = 256

In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# func baraye pish pardazeshe BART

def bart_preprocess(batch):
    """
    Tokenize Text-to-SQL inputs and their corresponding target SQL queries
    for BART sequence-to-sequence training.

    Args:
        batch (dict): Batch containing question, schema, and query fields.

    Returns:
        dict: Tokenized model inputs with target token IDs stored as labels.
    """
    
    inputs = [build_bart_input(q, s) for q, s in zip(batch["question"], batch["schema"])]
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LEN,
        truncation=True,
        padding=False,
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["query"],
            max_length=MAX_TARGET_LEN,
            truncation=True,
            padding=False,
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [15]:
# tokanization baraye train, dev, test

train_tok = train_ds_small.map(bart_preprocess, batched=True, remove_columns=train_ds_small.column_names)
dev_tok   = dev_ds_small.map(bart_preprocess, batched=True, remove_columns=dev_ds_small.column_names)
test_tok  = test_ds_small.map(bart_preprocess, batched=True, remove_columns=test_ds_small.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=MODEL_NAME)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

In [ ]:
if torch.cuda.is_available():
    model = model.cuda()

In [ ]:
# Training configuration

BATCH_SIZE = 8
GRAD_ACCUM = 2
LR = 5e-5
NUM_EPOCHS = 2

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_text2sql",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    fp16=torch.cuda.is_available(),
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
)


In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=dev_tok,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

# Train the model and measure runtime
start = time.time()
train_result = trainer.train()
elapsed = time.time() - start

/tmp/ipython-input-2507966957.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss
50,2.058400
100,1.340800
150,1.093700
200,0.953400
250,0.896700
300,0.878200
350,0.796900
400,0.772600
450,0.712900
500,0.706500


In [21]:
final_train_loss = train_result.metrics.get("train_loss", None)

In [22]:
print("\n===== TRAIN REPORT =====")
print("Model:", MODEL_NAME)
print("Train size:", len(train_ds_small))
print("Dev size:", len(dev_ds_small))
print("Epochs:", NUM_EPOCHS)
print("LR:", LR)
print("Batch size:", BATCH_SIZE, "GradAccum:", GRAD_ACCUM)
print("Max source len:", MAX_SOURCE_LEN, "Max target len:", MAX_TARGET_LEN)
print("Final train loss:", final_train_loss)
print("Train time (sec):", round(elapsed, 2))



===== TRAIN REPORT =====
Model: facebook/bart-base
Train size: 20000
Dev size: 2000
Epochs: 2
LR: 5e-05
Batch size: 8 GradAccum: 2
Max source len: 512 Max target len: 256
Final train loss: 0.6106011367797851
Train time (sec): 594.92


In [ ]:
def raw_em(pred, gold):
    """
    Compute Raw Exact Match between a predicted SQL query and its reference.

    Args:
        pred (str): Predicted SQL query.
        gold (str): Reference SQL query.

    Returns:
        int: 1 if the two strings match exactly, otherwise 0.
    """
    
    return int(pred == gold)

In [ ]:
def normalize_sql(sql):
    """
    Normalize a SQL query to reduce superficial formatting differences
    before exact-match evaluation.

    Args:
        sql (str): SQL query to normalize.

    Returns:
        str: Normalized SQL query.
    """
    
    if sql is None:
        return ""
    s = sql.strip()
    s = re.sub(r";+\s*$", "", s)

    s = sqlparse.format(
        s,
        keyword_case="lower",
        identifier_case=None,
        strip_comments=True,
        use_space_around_operators=True,
        reindent=False,
    )

    s = re.sub(r"\s+", " ", s).strip()

    return s

def normalized_em(pred, gold):
    """
    Compute Exact Match after normalizing both predicted and reference SQL.

    Args:
        pred (str): Predicted SQL query.
        gold (str): Reference SQL query.

    Returns:
        int: 1 if the normalized queries match, otherwise 0.
    """
    
    return int(normalize_sql(pred) == normalize_sql(gold))


In [ ]:
@torch.no_grad()
def generate_sql(dataset_small, max_new_tokens=MAX_TARGET_LEN, num_beams=4):
    """
    Generate SQL queries for all examples in a dataset using the fine-tuned
    BART model.

    Args:
        dataset_small (Dataset): Dataset containing question, schema,
            and reference query fields.
        max_new_tokens (int): Maximum number of tokens generated per query.
        num_beams (int): Number of beams used during beam-search decoding.

    Returns:
        tuple[list[str], list[str]]: Generated SQL queries and their
        corresponding reference queries.
    """
    
    model.eval()
    preds = []
    golds = []
    for ex in dataset_small:
        inp = build_bart_input(ex["question"], ex["schema"])
        enc = tokenizer(
            inp,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SOURCE_LEN
        )
        if torch.cuda.is_available():
            enc = {k: v.cuda() for k, v in enc.items()}

        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            early_stopping=True,
        )
        pred = tokenizer.decode(out[0], skip_special_tokens=True).strip()
        preds.append(pred)
        golds.append(ex["query"].strip())
    return preds, golds

def eval_em(dataset_small, name="dev"):
    """
    Evaluate a dataset using Raw Exact Match and Normalized Exact Match.

    Args:
        dataset_small (Dataset): Dataset to evaluate.
        name (str): Name of the evaluated data split.

    Returns:
        tuple: Predictions, reference queries, Raw EM, and Normalized EM.
    """
    
    preds, golds = generate_sql(dataset_small)

    raw = np.mean([raw_em(p, g) for p, g in zip(preds, golds)])
    norm = np.mean([normalized_em(p, g) for p, g in zip(preds, golds)])

    print(f"\n===== {name.upper()} EVAL =====")
    print("Raw EM:", round(raw, 4))
    print("Normalized EM:", round(norm, 4))
    return preds, golds, raw, norm


In [26]:
dev_preds, dev_golds, dev_raw, dev_norm = eval_em(dev_ds_small, "dev")


===== DEV EVAL =====
Raw EM: 0.224
Normalized EM: 0.2305


In [27]:
test_preds, test_golds, test_raw, test_norm = eval_em(test_ds_small, "test")


===== TEST EVAL =====
Raw EM: 0.2205
Normalized EM: 0.223


In [ ]:
def error_tags(pred, gold):
    """
    Identify simple structural differences between a predicted SQL query
    and its reference query.

    Args:
        pred (str): Predicted SQL query.
        gold (str): Reference SQL query.

    Returns:
        list[str]: Detected error categories.
    """
    
    tags = []
    npred, ngold = normalize_sql(pred), normalize_sql(gold)

    if npred == ngold:
        return ["correct(normalized)"]

    def has_kw(s, kw): 
        """
        Check whether a SQL keyword appears as a standalone token.

        Args:
            s (str): Normalized SQL string.
            kw (str): SQL keyword to search for.

        Returns:
            bool: True if the keyword is present, otherwise False.
        """
        return re.search(rf"\b{kw}\b", s) is not None

    for kw in ["select", "from", "where", "group by", "order by", "join", "limit"]:
        if " " in kw:
            if (kw in npred) != (kw in ngold):
                tags.append(f"missing_or_extra:{kw}")
        else:
            if has_kw(npred, kw) != has_kw(ngold, kw):
                tags.append(f"missing_or_extra:{kw}")

    for agg in ["count", "avg", "sum", "min", "max"]:
        if has_kw(npred, agg) != has_kw(ngold, agg):
            tags.append(f"agg_mismatch:{agg}")

    if len(tags) == 0:
        tags.append("other_structure_or_column_value_mismatch")

    return tags

In [ ]:
print("\n===== 5 RANDOM DEV SAMPLES (with analysis) =====")
idxs = random.sample(range(len(dev_ds_small)), 5)
for i in idxs:
    ex = dev_ds_small[i]
    pred = dev_preds[i]
    gold = ex["query"]

    print("\n------------------------------")
    print("Question:\n", ex["question"])
    print("\nSchema:\n", ex["schema"][:800], "..." if len(ex["schema"]) > 800 else "")  
    print("\nModel output:\n", pred)
    print("\nGold SQL:\n", gold)

    print("\nRaw EM:", raw_em(pred, gold))
    print("Normalized EM:", normalized_em(pred, gold))
    print("Error tags:", error_tags(pred, gold))

    print("\nNormalized pred:\n", normalize_sql(pred))
    print("\nNormalized gold:\n", normalize_sql(gold))



===== 5 RANDOM DEV SAMPLES (with analysis) =====

------------------------------
Question:
 Count the number of marine research projects in the Arctic and Southern Oceans per year.

Schema:
 CREATE TABLE research_projects (ocean TEXT, project_count INT, year INT); INSERT INTO research_projects (ocean, project_count, year) VALUES ('Arctic', 100, 2020), ('Arctic', 120, 2021), ('Southern', 150, 2020), ('Southern', 180, 2021); 

Model output:
 SELECT year, COUNT(project_count) FROM research_projects WHERE ocean IN ('Arctic', 'Southern') GROUP BY year;

Gold SQL:
 SELECT ocean, year, COUNT(*) FROM research_projects WHERE ocean IN ('Arctic', 'Southern') GROUP BY ocean, year;

Raw EM: 0
Normalized EM: 0
Error tags: ['other_structure_or_column_value_mismatch']

Normalized pred:
 select year, COUNT(project_count) from research_projects where ocean in ('Arctic', 'Southern') group by year

Normalized gold:
 select ocean, year, COUNT(*) from research_projects where ocean in ('Arctic', 'Southern')

# 3. Decoder-Only Fine-Tuning with GPT-2

## GPT-2 for Text-to-SQL

GPT-2 is a Decoder-Only Transformer architecture that generates text autoregressively by predicting the next token based on the preceding context.

For Text-to-SQL generation, the natural-language question and database schema are provided as a structured prefix, and the model learns to continue the sequence by generating the corresponding SQL query.

Unlike BART, which separately encodes the input and generates the target sequence through a decoder, GPT-2 represents the entire Text-to-SQL task within a single causal sequence.

---

## Input Construction

Each example is formatted as a single sequence containing:

```text
Question + Database Schema + SQL Prefix
```

The model then continues this prefix by generating the target SQL query.

A structured input format is used so that the model can distinguish between the user question, schema information, and SQL generation boundary.

Conceptually, the sequence follows the form:

```text
Question: ...
Schema: ...
SQL: ...
```

---

## Causal Language Modeling Pipeline

For training, each example is represented as:

```text
Prefix + Reference SQL + EOS
```

The complete sequence is tokenized for causal language modeling.

To ensure that the model is optimized specifically for SQL generation, the tokens belonging to the input prefix are masked in the training labels.

As a result, the loss is calculated only over the target SQL portion of the sequence.

The preprocessing pipeline therefore includes:

* Constructing the question-and-schema prefix
* Appending the reference SQL query
* Adding the end-of-sequence token
* Tokenizing the complete sequence
* Creating attention masks
* Masking prefix tokens in the labels

---

## Model Training

GPT-2 is fine-tuned as a causal language model on the prepared Text-to-SQL examples.

The training process records the main experimental settings and outputs, including:

* Training loss
* Training runtime
* Learning rate
* Batch size
* Number of epochs
* Maximum sequence length

---

## SQL Generation

During inference, only the input prefix is provided to the model:

```text
Question + Database Schema + SQL:
```

GPT-2 then autoregressively generates the continuation.

Because the generated sequence may contain both the original prefix and the generated continuation, the prefix is removed before evaluating the predicted SQL query.

Only the generated SQL portion is used for comparison with the reference query.

---

## Evaluation

The fine-tuned GPT-2 model is evaluated independently on the development and test subsets using the same metrics applied to BART:

* Raw Exact Match
* Normalized Exact Match

Using identical evaluation criteria allows the two architectures to be compared under a consistent experimental setup.

---

## Qualitative Error Analysis

Several development examples are inspected manually to examine the generation behavior of GPT-2.

For each example, the analysis includes:

* Natural-language question
* Database schema
* Generated SQL query
* Reference SQL query
* Raw Exact Match result
* Normalized Exact Match result
* Observed generation or structural errors

This analysis is used to identify recurring failure patterns and later compare GPT-2 with the Encoder–Decoder BART model.


In [ ]:
def build_gpt2_prefix(question, schema):
    """
    Construct the GPT-2 input prefix from a natural-language question
    and its associated database schema.

    Args:
        question (str): Natural-language question to be translated into SQL.
        schema (str): Database schema associated with the question.

    Returns:
        str: Formatted prefix used as context for SQL generation.
    """
    
    return f"question: {question}\nschema: {schema}\nSQL:"


In [ ]:
def build_gpt2_full_text(question: str, schema: str, gold_sql: str, eos_token: str) -> str:
    """
    Construct the complete causal language modeling sequence containing
    the input prefix, reference SQL query, and end-of-sequence token.

    Args:
        question (str): Natural-language question.
        schema (str): Database schema associated with the question.
        gold_sql (str): Reference SQL query.
        eos_token (str): End-of-sequence token used by the tokenizer.

    Returns:
        str: Complete training sequence for GPT-2.
    """
    
    prefix = build_gpt2_prefix(question, schema)
    return f"{prefix}\n{gold_sql.strip()}{eos_token}"


In [ ]:
MODEL_NAME = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
if torch.cuda.is_available():
    model = model.cuda()

MAX_LEN = 768 
EOS = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
def gpt2_preprocess(batch):
    """
    Prepare Text-to-SQL examples for GPT-2 causal language modeling.

    The question and schema form the input prefix, while the reference SQL
    query is appended as the target continuation. Prefix tokens are masked
    in the labels so that they do not contribute to the training loss.

    Args:
        batch (dict): Batch containing question, schema, and query fields.

    Returns:
        dict: Tokenized input IDs, attention masks, and masked labels.
    """
    
    input_ids_list = []
    attn_list = []
    labels_list = []

    for q, s, sql in zip(batch["question"], batch["schema"], batch["query"]):
        prefix = build_gpt2_prefix(q, s)
        full_text = build_gpt2_full_text(q, s, sql, EOS)

        pref_ids = tokenizer(prefix, add_special_tokens=False)["input_ids"]
        full = tokenizer(
            full_text,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_LEN,
        )
        ids = full["input_ids"]
        attn = full["attention_mask"]

        prefix_len = min(len(pref_ids), len(ids))

        labels = ids.copy()
        for i in range(prefix_len):
            labels[i] = -100

        input_ids_list.append(ids)
        attn_list.append(attn)
        labels_list.append(labels)

    return {"input_ids": input_ids_list, "attention_mask": attn_list, "labels": labels_list}

In [ ]:

train_gpt2 = train_ds_small.map(gpt2_preprocess, batched=True, remove_columns=train_ds_small.column_names)
dev_gpt2   = dev_ds_small.map(gpt2_preprocess, batched=True, remove_columns=dev_ds_small.column_names)
test_gpt2  = test_ds_small.map(gpt2_preprocess, batched=True, remove_columns=test_ds_small.column_names)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
def pad_collate(features):
    """
    Pad variable-length GPT-2 samples within a batch.

    Input IDs are padded with the tokenizer padding token, attention masks
    with zeros, and labels with -100 so padded positions are ignored
    during loss computation.

    Args:
        features (list[dict]): Tokenized samples in a training batch.

    Returns:
        dict: Padded input IDs, attention masks, and labels.
    """
    
    batch_input_ids = [torch.tensor(f["input_ids"], dtype=torch.long) for f in features]
    batch_attn      = [torch.tensor(f["attention_mask"], dtype=torch.long) for f in features]
    batch_labels    = [torch.tensor(f["labels"], dtype=torch.long) for f in features]

    input_ids = torch.nn.utils.rnn.pad_sequence(batch_input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attn      = torch.nn.utils.rnn.pad_sequence(batch_attn, batch_first=True, padding_value=0)
    labels    = torch.nn.utils.rnn.pad_sequence(batch_labels, batch_first=True, padding_value=-100)

    return {"input_ids": input_ids, "attention_mask": attn, "labels": labels}


In [ ]:
# Training configuration
training_args = TrainingArguments(
    output_dir="./gpt2_text2sql",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    fp16=torch.cuda.is_available(),
    save_strategy="no",
    logging_steps=50,
    report_to="none",
)

In [38]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_gpt2,
    eval_dataset=dev_gpt2,
    data_collator=pad_collate,
    tokenizer=tokenizer,
)

/tmp/ipython-input-2818188865.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# Train the model and measure runtime
start = time.time()
train_result = trainer.train()
elapsed = time.time() - start

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,1.371700
100,1.015600
150,0.871400
200,0.787000
250,0.743700
300,0.748700
350,0.687000
400,0.667400
450,0.637900
500,0.611700


In [40]:
final_train_loss = train_result.metrics.get("train_loss", None)

In [41]:
print("\n===== TRAIN REPORT (GPT-2) =====")
print("Model:", MODEL_NAME)
print("Train size:", len(train_ds_small))
print("Dev size:", len(dev_ds_small))
print("Epochs:", NUM_EPOCHS)
print("LR:", LR)
print("Batch size:", BATCH_SIZE, "GradAccum:", GRAD_ACCUM)
print("MAX_LEN:", MAX_LEN)
print("Final train loss:", final_train_loss)
print("Train time (sec):", round(elapsed, 2))


===== TRAIN REPORT (GPT-2) =====
Model: gpt2
Train size: 20000
Dev size: 2000
Epochs: 2
LR: 5e-05
Batch size: 4 GradAccum: 4
MAX_LEN: 768
Final train loss: 0.5380672233581543
Train time (sec): 1156.31


In [ ]:
@torch.no_grad()
def generate_sql_gpt2(dataset_small, max_new_tokens=128, do_sample=False, num_beams=4,
                      temperature=1.0, top_p=0.9):
    """
    Generate SQL queries from question-schema prefixes using GPT-2.

    The generated continuation is separated from the original input prefix
    before decoding so that only the predicted SQL is returned.

    Args:
        dataset_small (Dataset): Dataset containing question, schema,
            and reference query fields.
        max_new_tokens (int): Maximum number of newly generated tokens.
        do_sample (bool): Whether to use stochastic sampling.
        num_beams (int): Number of beams used for deterministic generation.
        temperature (float): Sampling temperature when sampling is enabled.
        top_p (float): Nucleus sampling probability threshold.

    Returns:
        tuple[list[str], list[str]]: Generated SQL queries and their
        corresponding reference queries.
    """
    
    model.eval()
    preds, golds = [], []

    for ex in dataset_small:
        prefix = build_gpt2_prefix(ex["question"], ex["schema"])
        enc = tokenizer(
            prefix,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LEN,
            padding=False,
        )

        if torch.cuda.is_available():
            enc = {k: v.cuda() for k, v in enc.items()}

        input_len = enc["input_ids"].shape[1]

        gen_kwargs = dict(
            **enc,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        if do_sample:
            gen_kwargs.update(dict(do_sample=True, num_beams=1, temperature=temperature, top_p=top_p))

        else:
            gen_kwargs.update(dict(
                do_sample=False,
                num_beams=num_beams,
                early_stopping=True,
            ))

        out = model.generate(**gen_kwargs)

        gen_ids = out[0, input_len:]
        pred_sql = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

        preds.append(pred_sql)
        golds.append(ex["query"].strip())

    return preds, golds


In [ ]:
def eval_em_gpt2(dataset_small, name="dev", **gen_kwargs):
    """
    Evaluate GPT-2 SQL generation using Raw Exact Match and
    Normalized Exact Match.

    Args:
        dataset_small (Dataset): Dataset split to evaluate.
        name (str): Name of the evaluated split.
        **gen_kwargs: Additional generation arguments passed to
            generate_sql_gpt2.

    Returns:
        tuple: Predictions, reference queries, Raw EM, and Normalized EM.
    """
    preds, golds = generate_sql_gpt2(dataset_small, **gen_kwargs)

    raw = float(np.mean([raw_em(p, g) for p, g in zip(preds, golds)]))
    norm = float(np.mean([normalized_em(p, g) for p, g in zip(preds, golds)]))

    print(f"\n===== {name.upper()} EVAL (GPT-2) =====")
    print("Raw EM:", round(raw, 4))
    print("Normalized EM:", round(norm, 4))
    return preds, golds, raw, norm


In [44]:
dev_preds, dev_golds, dev_raw, dev_norm = eval_em_gpt2(dev_ds_small, "dev")
test_preds, test_golds, test_raw, test_norm = eval_em_gpt2(test_ds_small, "test")


===== DEV EVAL (GPT-2) =====
Raw EM: 0.18
Normalized EM: 0.187

===== TEST EVAL (GPT-2) =====
Raw EM: 0.185
Normalized EM: 0.1895


In [ ]:
print("\n===== 5 RANDOM DEV SAMPLES (GPT-2) =====")
idxs = random.sample(range(len(dev_ds_small)), 5)
for i in idxs:
    ex = dev_ds_small[i]
    pred = dev_preds[i]
    gold = ex["query"]

    print("\n------------------------------")
    print("Question:\n", ex["question"])
    print("\nSchema:\n", ex["schema"][:800], "..." if len(ex["schema"]) > 800 else "")
    print("\nModel output (SQL only):\n", pred)
    print("\nGold SQL:\n", gold)

    print("\nRaw EM:", raw_em(pred, gold))
    print("Normalized EM:", normalized_em(pred, gold))

    if "error_tags" in globals():
        print("Error tags:", error_tags(pred, gold))

    print("\nNormalized pred:\n", normalize_sql(pred))
    print("\nNormalized gold:\n", normalize_sql(gold))



===== 5 RANDOM DEV SAMPLES (GPT-2) =====

------------------------------
Question:
 Count the number of marine research projects in the Arctic and Southern Oceans per year.

Schema:
 CREATE TABLE research_projects (ocean TEXT, project_count INT, year INT); INSERT INTO research_projects (ocean, project_count, year) VALUES ('Arctic', 100, 2020), ('Arctic', 120, 2021), ('Southern', 150, 2020), ('Southern', 180, 2021); 

Model output (SQL only):
 SELECT year, COUNT(*) FROM research_projects GROUP BY year;

Gold SQL:
 SELECT ocean, year, COUNT(*) FROM research_projects WHERE ocean IN ('Arctic', 'Southern') GROUP BY ocean, year;

Raw EM: 0
Normalized EM: 0
Error tags: ['missing_or_extra:where']

Normalized pred:
 select year, COUNT(*) from research_projects group by year

Normalized gold:
 select ocean, year, COUNT(*) from research_projects where ocean in ('Arctic', 'Southern') group by ocean, year

------------------------------
Question:
 Delete all records of non-sustainable fabric types